In [ ]:
%%writefile flash_v10.cu
#include<iostream>
#include<cuda_runtime.h>
using namespace std;

#define TILE_SIZE 32

__global__ void flashAttention_v10(float *Q, float *K, float *V, float *O, int N , int d){
  // warp & lane identification
  int warp_id = threadIdx.x / 32;
  int lane_id = threadIdx.x % 32;
  int warps_per_block = blockDim.x / 32;

  int i = blockIdx.x * warps_per_block + warp_id;

  if(i >= N) return;

  float scale = rsqrtf((float)d);

  // registers tiling
  float qi_local[2];
  float oi_local[2] = {0.0f};


  #pragma unroll
  for(int k = 0; k < 2; k++){
    int col = lane_id + k * 32;
    if(col < d){
      qi_local[k] = Q[i * d + col] * scale;
    }
  }

  float m = -INFINITY;
  float l = 0.0f;

  __shared__ float K_tile[TILE_SIZE][64];
  __shared__ float V_tile[TILE_SIZE][64];

  // main Loop
  for(int tile_start = 0; tile_start < N; tile_start += TILE_SIZE){
    int tid = threadIdx.x;

    for(int idx = tid; idx < (TILE_SIZE * d) / 4; idx += blockDim.x){

      int base = idx * 4;
      int row = base >> 6;
      int col = base & 63;

      if((tile_start + row) < N){
        float4 k_val = reinterpret_cast<float4*>(&K[(tile_start + row) * d + col])[0];
        float4 v_val = reinterpret_cast<float4*>(&V[(tile_start + row) * d + col])[0];

        K_tile[row][col] = k_val.x;
        K_tile[row][col + 1] = k_val.y;
        K_tile[row][col + 2] = k_val.z;
        K_tile[row][col + 3] = k_val.w;

        V_tile[row][col] = v_val.x;
        V_tile[row][col + 1] = v_val.y;
        V_tile[row][col + 2] = v_val.z;
        V_tile[row][col + 3] = v_val.w;
      }
    }
    __syncthreads();

    int valid_j = min(TILE_SIZE , N - tile_start);

    for(int j = 0; j < valid_j; j += 4){
      float scores[4] = {0.0f, 0.0f, 0.0f, 0.0f};

      #pragma unroll
      for(int k = 0; k < 2; k++){
        int col = lane_id + k * 32;
        if(col < d){
          scores[0] += qi_local[k] * K_tile[j][col];

          if(j+1 < valid_j) scores[1] += qi_local[k] * K_tile[j+1][col];
          if(j+2 < valid_j) scores[2] += qi_local[k] * K_tile[j+2][col];
          if(j+3 < valid_j) scores[3] += qi_local[k] * K_tile[j+3][col];
        }
      }

      // warp reduction
      for(int offset = 16; offset > 0; offset /= 2){
        for(int t =0; t < 4; t++){
          scores[t] = __shfl_down_sync(0xffffffff, scores[t] , offset);
        }
      }

      for(int t=0; t<4; t++)
        scores[t] = __shfl_sync(0xffffffff, scores[t], 0);

      float m_old = m;

      //sequential softmax
      if(lane_id == 0){
        #pragma unroll
        for(int t=0; t<4; t++){
          if(j + t < valid_j){
            float m_prev = m;
            m = fmaxf(m_prev, scores[t]);
            float p = __expf(scores[t] - m);
            l = l * __expf(m_prev - m) + p;
          }
        }
      }

      m = __shfl_sync(0xffffffff, m, 0);
      l = __shfl_sync(0xffffffff, l, 0);

      #pragma unroll
      for(int k = 0; k < 2; k++){
        oi_local[k] *= __expf(m_old - m);
      }
      // update accumalation
      #pragma unroll
      for(int k = 0; k < 2; k++){
        int col = lane_id + k * 32;

        if(col<d){
          for(int t= 0; t< 4; t++){
            if(j + t < valid_j){
              float v = V_tile[j+t][col];
              float p = __expf(scores[t] - m);
              oi_local[k] = oi_local[k] + p * v;
            }
          }
        }
      }
    }
    __syncthreads();
  }
  #pragma unroll
  for(int k = 0; k < 2; k++){
    int col = lane_id + k * 32;
    if(col < d)
      O[i * d + col] = oi_local[k] / l;
  }
}



#define CUDA_CHECK(call)                                      \
{                                                             \
    cudaError_t err = call;                                   \
    if (err != cudaSuccess) {                                 \
        cout << "CUDA error: " << cudaGetErrorString(err)     \
             << " at " << __FILE__ << ":" << __LINE__ << endl;\
        exit(1);                                              \
    }                                                         \
}



// host code
int main(){
  int N = 8192;
  int d = 64;

  size_t size = N * d * sizeof(float);

  //variables host + device
  float *h_Q, *h_K, *h_V, *h_O;
  float *d_Q, *d_K, *d_V, *d_O;

  // host memory allocation
  h_Q = (float*)malloc(size);
  h_K = (float*)malloc(size);
  h_V = (float*)malloc(size);
  h_O = (float*)malloc(size);

  // data initialization
  for(int i = 0; i < N* d; i++){
    h_Q[i] = static_cast<float>(rand()) / RAND_MAX;
    h_K[i] = static_cast<float>(rand()) / RAND_MAX;
    h_V[i] = static_cast<float>(rand()) / RAND_MAX;
  }

  // device memory allocation
  CUDA_CHECK(cudaMalloc((void**)&d_Q, size));
  CUDA_CHECK(cudaMalloc((void**)&d_K, size));
  CUDA_CHECK(cudaMalloc((void**)&d_V, size));
  CUDA_CHECK(cudaMalloc((void**)&d_O, size));

  //Copy data from host ->> to device
  CUDA_CHECK(cudaMemcpy(d_Q, h_Q , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_K, h_K , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_V, h_V , size , cudaMemcpyHostToDevice));


  // kernel config
  int threads = 256;
  int warps_per_block = threads / 32;
  int block = (N + warps_per_block - 1) / warps_per_block;

  // warmups
  for(int i = 0; i < 100; i++){
    flashAttention_v10<<<block , threads>>>(d_Q , d_K , d_V , d_O , N , d);
  }
  cudaDeviceSynchronize();

  //timing measurement
  cudaEvent_t start , stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);

  const int TIMED_RUNS = 100;
  for(int i = 0; i < TIMED_RUNS; i++){
    flashAttention_v10<<<block , threads>>>(d_Q, d_K, d_V, d_O , N , d);
  }

  cudaEventRecord(stop);
  cudaEventSynchronize(stop);


  // timing
  float ms = 0;
  cudaEventElapsedTime(&ms , start , stop);
  ms /= TIMED_RUNS;

  // FLOPs
  double flops = 4.0 * N * N * d;

  // Bytes (approx global memory traffic) or Throughput
  double bytes = 3.0 * N * N * sizeof(float);  // rough estimate

  // Metrics
  double gflops = (flops / (ms / 1000.0)) / 1e9;
  double bandwidth = (bytes / (ms / 1000.0)) / 1e9;
  double intensity = flops / bytes;

  cout << "Time: " << ms << " ms\n";
  cout << "GFLOPS: " << gflops << "\n";
  cout << "Memory Throughput (GB/s): " << bandwidth << "\n";
  cout << "Arithmetic Intensity: " << intensity << "\n";

  printf("METRIC,time_ms,%.4f\n", ms);
  printf("METRIC,gflops,%.2f\n", gflops);
  printf("METRIC,bandwidth,%.2f\n", bandwidth);
  printf("METRIC,intensity,%.2f\n", intensity);

  //cleanup
  cudaFree(d_Q);
  cudaFree(d_K);
  cudaFree(d_V);
  cudaFree(d_O);

  free(h_Q);
  free(h_K);
  free(h_V);
  free(h_O);

  return 0;
}


Overwriting flash_v10.cu


In [ ]:
!nvcc flash_v10.cu -o flash_v10 -arch=sm_75

In [ ]:
!./flash_v10

Time: 33.8601 ms
GFLOPS: 507.377
Memory Throughput (GB/s): 23.7833
Arithmetic Intensity: 21.3333
METRIC,time_ms,33.8601
METRIC,gflops,507.38
METRIC,bandwidth,23.78
METRIC,intensity,21.33


In [ ]:
result = subprocess.check_output([
    "ncu",
    "--launch-count", "1",
    "--launch-skip", "100",

    "--section", "WarpStateStats",
    "--section", "MemoryWorkloadAnalysis",
    "--section", "Occupancy",
    "--section", "SpeedOfLight",

    "./flash_v10"
], text=True, stderr=subprocess.STDOUT)

print(result)

==PROF== Connected to process 2309 (/content/flash_v10)
==PROF== Profiling "flashAttention_v10" - 0 (1/1): 0%....50%....100% - 12 passes
Time: 80.6876 ms
GFLOPS: 212.918
Memory Throughput (GB/s): 9.98055
Arithmetic Intensity: 21.3333
METRIC,time_ms,80.6876
METRIC,gflops,212.92
METRIC,bandwidth,9.98
METRIC,intensity,21.33
==PROF== Disconnected from process 2309
[2309] flash_v10@127.0.0.1
  flashAttention_v10(float *, float *, float *, float *, int, int) (1024, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- -------------
    Metric Name             Metric Unit  Metric Value
    ----------------------- ----------- -------------
    DRAM Frequency                  Ghz          5.00
    SM Frequency                    Mhz        585.00
    Elapsed Cycles                cycle    42,351,459
    Memory Throughput                 %         85.17
    DRAM Throughput                   %          0.23
    

### PHASE- 11 - FLASHATTETNION - V11

  * Optimization 1: Vectorized Shared Memory Writes
    * In v11, the kernel uses reinterpret_cast<float4*> on both sides of the equation:
      * This tells the GPU compile-time architecture to execute a single, wide 128-bit store instruction straight into SRAM (Shared Memory). It drastically improves execution efficiency and reduces instruction overhead.
  * Optimization 2: Splitting into a "Fast Path" and a "Tail Path"
    * In v10, the loop constantly checked branches inside the inner loop: if(j+1 < valid_j). Branching inside an unrolled loop slows down execution because threads have to evaluate conditionals repeatedly.


  * `IMPORTANT STUFF`
    * What is valid_j?
      * How many VALID rows exist in this tile? we use it because last row of the tile may be not full filled
      N = 100
      at the last 96 = 96-99
      only 4 rows exits instead of 32


    * What is full_j?
      * “Largest multiple of 4 that fits safely.”

In [ ]:
%%writefile flash_v11.cu
#include<iostream>
#include<cuda_runtime.h>
using namespace std;

#define TILE_SIZE 32


__global__ void flashAttention_v11(float *Q, float *K, float *V,
                                   float *O, int N, int d){

    int warp_id = threadIdx.x / 32;
    int lane_id = threadIdx.x % 32;
    int warps_per_block = blockDim.x / 32;

    int i = blockIdx.x * warps_per_block + warp_id;

    if(i >= N) return;

    constexpr int D = 64;

    float scale = rsqrtf((float)D);

    // register tiles
    float qi_local[2];
    float oi_local[2] = {0.0f, 0.0f};

    #pragma unroll
    for(int k = 0; k < 2; k++){
        int col = lane_id + k * 32;
        qi_local[k] = Q[i * D + col] * scale;
    }

    float m = -INFINITY;
    float l = 0.0f;

    __shared__ float K_tile[TILE_SIZE][D];
    __shared__ float V_tile[TILE_SIZE][D];

    for(int tile_start = 0; tile_start < N; tile_start += TILE_SIZE){

        int tid = threadIdx.x;

        // vectorized loads
        for(int idx = tid; idx < (TILE_SIZE * D) / 4; idx += blockDim.x){

            int base = idx * 4;
            int row = base >> 6;
            int col = base & 63;

            if((tile_start + row) < N){

                float4 k_val =
                    reinterpret_cast<float4*>(
                        &K[(tile_start + row) * D + col])[0];

                float4 v_val =
                    reinterpret_cast<float4*>(
                        &V[(tile_start + row) * D + col])[0];

                reinterpret_cast<float4*>(&K_tile[row][col])[0] = k_val;
                reinterpret_cast<float4*>(&V_tile[row][col])[0] = v_val;
            }
        }

        __syncthreads();

        int valid_j = min(TILE_SIZE, N - tile_start);

        // =========================
        // FULL TILE FAST PATH
        // =========================

        int full_j = (valid_j / 4) * 4;

        for(int j = 0; j < full_j; j += 4){

            float scores[4] = {0.f, 0.f, 0.f, 0.f};

            #pragma unroll
            for(int k = 0; k < 2; k++){

                int col = lane_id + k * 32;

                float q = qi_local[k];

                scores[0] += q * K_tile[j][col];
                scores[1] += q * K_tile[j+1][col];
                scores[2] += q * K_tile[j+2][col];
                scores[3] += q * K_tile[j+3][col];
            }

            // warp reduction
            for(int offset = 16; offset > 0; offset /= 2){

                #pragma unroll
                for(int t = 0; t < 4; t++){
                    scores[t] += __shfl_down_sync(
                        0xffffffff, scores[t], offset);
                }
            }

            #pragma unroll
            for(int t = 0; t < 4; t++){
                scores[t] = __shfl_sync(0xffffffff, scores[t], 0);
            }

            float m_old = m;

            // sequential softmax bookkeeping
            if(lane_id == 0){

                #pragma unroll
                for(int t = 0; t < 4; t++){

                    float m_prev = m;

                    m = fmaxf(m_prev, scores[t]);

                    float p = __expf(scores[t] - m);

                    l = l * __expf(m_prev - m) + p;
                }
            }

            m = __shfl_sync(0xffffffff, m, 0);
            l = __shfl_sync(0xffffffff, l, 0);

            float scale_old = __expf(m_old - m);

            #pragma unroll
            for(int k = 0; k < 2; k++){
                oi_local[k] *= scale_old;
            }

            #pragma unroll
            for(int k = 0; k < 2; k++){

                int col = lane_id + k * 32;

                float q_acc = oi_local[k];

                #pragma unroll
                for(int t = 0; t < 4; t++){

                    float p = __expf(scores[t] - m);

                    q_acc += p * V_tile[j+t][col];
                }

                oi_local[k] = q_acc;
            }
        }

        // =========================
        // TAIL PATH
        // =========================

        for(int j = full_j; j < valid_j; j++){

            float score = 0.f;

            #pragma unroll
            for(int k = 0; k < 2; k++){

                int col = lane_id + k * 32;

                score += qi_local[k] * K_tile[j][col];
            }

            for(int offset = 16; offset > 0; offset /= 2){
                score += __shfl_down_sync(
                    0xffffffff, score, offset);
            }

            score = __shfl_sync(0xffffffff, score, 0);

            float m_old = m;

            if(lane_id == 0){

                float m_prev = m;

                m = fmaxf(m_prev, score);

                float p = __expf(score - m);

                l = l * __expf(m_prev - m) + p;
            }

            m = __shfl_sync(0xffffffff, m, 0);
            l = __shfl_sync(0xffffffff, l, 0);

            float scale_old = __expf(m_old - m);

            #pragma unroll
            for(int k = 0; k < 2; k++){
                oi_local[k] *= scale_old;
            }

            #pragma unroll
            for(int k = 0; k < 2; k++){

                int col = lane_id + k * 32;

                float p = __expf(score - m);

                oi_local[k] += p * V_tile[j][col];
            }
        }

        __syncthreads();
    }

    #pragma unroll
    for(int k = 0; k < 2; k++){

        int col = lane_id + k * 32;

        O[i * D + col] = oi_local[k] / l;
    }
}





#define CUDA_CHECK(call)                                      \
{                                                             \
    cudaError_t err = call;                                   \
    if (err != cudaSuccess) {                                 \
        cout << "CUDA error: " << cudaGetErrorString(err)     \
             << " at " << __FILE__ << ":" << __LINE__ << endl;\
        exit(1);                                              \
    }                                                         \
}



// host code
int main(){
  int N = 8192;
  int d = 64;

  size_t size = N * d * sizeof(float);

  //variables host + device
  float *h_Q, *h_K, *h_V, *h_O;
  float *d_Q, *d_K, *d_V, *d_O;

  // host memory allocation
  h_Q = (float*)malloc(size);
  h_K = (float*)malloc(size);
  h_V = (float*)malloc(size);
  h_O = (float*)malloc(size);

  // data initialization
  for(int i = 0; i < N* d; i++){
    h_Q[i] = static_cast<float>(rand()) / RAND_MAX;
    h_K[i] = static_cast<float>(rand()) / RAND_MAX;
    h_V[i] = static_cast<float>(rand()) / RAND_MAX;
  }

  // device memory allocation
  CUDA_CHECK(cudaMalloc((void**)&d_Q, size));
  CUDA_CHECK(cudaMalloc((void**)&d_K, size));
  CUDA_CHECK(cudaMalloc((void**)&d_V, size));
  CUDA_CHECK(cudaMalloc((void**)&d_O, size));

  //Copy data from host ->> to device
  CUDA_CHECK(cudaMemcpy(d_Q, h_Q , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_K, h_K , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_V, h_V , size , cudaMemcpyHostToDevice));


  // kernel config
  int threads = 256;
  int warps_per_block = threads / 32;
  int block = (N + warps_per_block - 1) / warps_per_block;

  // warmups
  for(int i = 0; i < 100; i++){
    flashAttention_v11<<<block , threads>>>(d_Q , d_K , d_V , d_O , N , d);
  }
  cudaDeviceSynchronize();

  //timing measurement
  cudaEvent_t start , stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);

  const int TIMED_RUNS = 100;
  for(int i = 0; i < TIMED_RUNS; i++){
    flashAttention_v11<<<block , threads>>>(d_Q, d_K, d_V, d_O , N , d);
  }

  cudaEventRecord(stop);
  cudaEventSynchronize(stop);


  // timing
  float ms = 0;
  cudaEventElapsedTime(&ms , start , stop);
  ms /= TIMED_RUNS;

  // FLOPs
  double flops = 4.0 * N * N * d;

  // Bytes (approx global memory traffic) or Throughput
  double bytes = 3.0 * N * N * sizeof(float);  // rough estimate

  // Metrics
  double gflops = (flops / (ms / 1000.0)) / 1e9;
  double bandwidth = (bytes / (ms / 1000.0)) / 1e9;
  double intensity = flops / bytes;

  cout << "Time: " << ms << " ms\n";
  cout << "GFLOPS: " << gflops << "\n";
  cout << "Memory Throughput (GB/s): " << bandwidth << "\n";
  cout << "Arithmetic Intensity: " << intensity << "\n";

  printf("METRIC,time_ms,%.4f\n", ms);
  printf("METRIC,gflops,%.2f\n", gflops);
  printf("METRIC,bandwidth,%.2f\n", bandwidth);
  printf("METRIC,intensity,%.2f\n", intensity);

  //cleanup
  cudaFree(d_Q);
  cudaFree(d_K);
  cudaFree(d_V);
  cudaFree(d_O);

  free(h_Q);
  free(h_K);
  free(h_V);
  free(h_O);

  return 0;
}

Writing flash_v11.cu


In [ ]:
!nvcc flash_v11.cu -o flash_v11 -arch=sm_75

In [ ]:
!./flash_v11

Time: 32.6793 ms
GFLOPS: 525.71
Memory Throughput (GB/s): 24.6427
Arithmetic Intensity: 21.3333
METRIC,time_ms,32.6793
METRIC,gflops,525.71
METRIC,bandwidth,24.64
METRIC,intensity,21.33


In [ ]:
%%writefile flash_v12.cu
#include<iostream>
#include<cuda_runtime.h>
using namespace std;



#define TILE_SIZE 32
#define D 64

__global__ void flashAttention_v12(
    float *Q,
    float *K,
    float *V,
    float *O,
    int N)
{
    // =========================
    // Warp mapping
    // =========================

    int warp_id  = threadIdx.x >> 5;
    int lane_id  = threadIdx.x & 31;

    int warps_per_block = blockDim.x >> 5;

    int i = blockIdx.x * warps_per_block + warp_id;

    if(i >= N) return;

    // =========================
    // Shared memory
    // =========================

    __shared__ float K_tile[TILE_SIZE][D];
    __shared__ float V_tile[TILE_SIZE][D];

    // =========================
    // Registers
    // =========================

    float scale = rsqrtf((float)D);

    float qi0 = Q[i * D + lane_id] * scale;            // This is classic HPC optimization.
    float qi1 = Q[i * D + lane_id + 32] * scale;

    float oi0 = 0.f;
    float oi1 = 0.f;

    float m = -INFINITY;
    float l = 0.f;

    // =========================
    // Main tile loop
    // =========================

    for(int tile_start = 0;
        tile_start < N;
        tile_start += TILE_SIZE)
    {
        // =========================
        // Vectorized global loads
        // =========================

        int tid = threadIdx.x;

        for(int idx = tid;
            idx < (TILE_SIZE * D) / 4;
            idx += blockDim.x)
        {
            int base = idx << 2;

            int row = base >> 6;
            int col = base & 63;

            if(tile_start + row < N)
            {
                float4 kv =
                    reinterpret_cast<float4*>(
                        &K[(tile_start + row) * D + col])[0];

                float4 vv =
                    reinterpret_cast<float4*>(
                        &V[(tile_start + row) * D + col])[0];

                reinterpret_cast<float4*>(
                    &K_tile[row][col])[0] = kv;

                reinterpret_cast<float4*>(
                    &V_tile[row][col])[0] = vv;
            }
        }

        __syncthreads();

        int valid_j =
            min(TILE_SIZE, N - tile_start);

        int full_j = (valid_j >> 2) << 2;

        // =========================================
        // FULL TILE PATH
        // =========================================

        for(int j = 0; j < full_j; j += 4)
        {
            // =========================
            // Register cached V values
            // =========================

            float v00 = V_tile[j][lane_id];
            float v01 = V_tile[j][lane_id + 32];

            float v10 = V_tile[j+1][lane_id];
            float v11 = V_tile[j+1][lane_id + 32];

            float v20 = V_tile[j+2][lane_id];
            float v21 = V_tile[j+2][lane_id + 32];

            float v30 = V_tile[j+3][lane_id];
            float v31 = V_tile[j+3][lane_id + 32];

            // =========================
            // Dot products
            // =========================

            float s0 =
                qi0 * K_tile[j][lane_id] +
                qi1 * K_tile[j][lane_id + 32];

            float s1 =
                qi0 * K_tile[j+1][lane_id] +
                qi1 * K_tile[j+1][lane_id + 32];

            float s2 =
                qi0 * K_tile[j+2][lane_id] +
                qi1 * K_tile[j+2][lane_id + 32];

            float s3 =
                qi0 * K_tile[j+3][lane_id] +
                qi1 * K_tile[j+3][lane_id + 32];

            // =========================
            // Warp reduction
            // =========================

            #pragma unroll
            for(int offset = 16;
                offset > 0;
                offset >>= 1)
            {
                s0 += __shfl_down_sync(
                    0xffffffff, s0, offset);

                s1 += __shfl_down_sync(
                    0xffffffff, s1, offset);

                s2 += __shfl_down_sync(
                    0xffffffff, s2, offset);

                s3 += __shfl_down_sync(
                    0xffffffff, s3, offset);
            }

            s0 = __shfl_sync(0xffffffff, s0, 0);
            s1 = __shfl_sync(0xffffffff, s1, 0);
            s2 = __shfl_sync(0xffffffff, s2, 0);
            s3 = __shfl_sync(0xffffffff, s3, 0);

            // =========================
            // Online softmax
            // =========================

            float m_old = m;

            float local_max =
                fmaxf(fmaxf(s0, s1),
                      fmaxf(s2, s3));

            float m_new = fmaxf(m, local_max);

            // =========================
            // Compute exponentials ONCE
            // =========================

            float p0 = __expf(s0 - m_new);
            float p1 = __expf(s1 - m_new);
            float p2 = __expf(s2 - m_new);
            float p3 = __expf(s3 - m_new);

            float l_new =
                l * __expf(m - m_new)
                + p0 + p1 + p2 + p3;

            m = m_new;
            l = l_new;

            float scale_old =
                __expf(m_old - m);

            // =========================
            // Rescale accumulators
            // =========================

            oi0 *= scale_old;
            oi1 *= scale_old;

            // =========================
            // Accumulate outputs
            // =========================

            oi0 +=
                p0 * v00 +
                p1 * v10 +
                p2 * v20 +
                p3 * v30;

            oi1 +=
                p0 * v01 +
                p1 * v11 +
                p2 * v21 +
                p3 * v31;
        }

        // =========================================
        // Tail path
        // =========================================

        for(int j = full_j;
            j < valid_j;
            j++)
        {
            float s =
                qi0 * K_tile[j][lane_id] +
                qi1 * K_tile[j][lane_id + 32];

            #pragma unroll
            for(int offset = 16;
                offset > 0;
                offset >>= 1)
            {
                s += __shfl_down_sync(
                    0xffffffff, s, offset);
            }

            s = __shfl_sync(0xffffffff, s, 0);

            float m_old = m;

            float m_new = fmaxf(m, s);

            float p = __expf(s - m_new);

            l =
                l * __expf(m - m_new)
                + p;

            m = m_new;

            float scale_old =
                __expf(m_old - m);

            oi0 *= scale_old;
            oi1 *= scale_old;

            oi0 += p * V_tile[j][lane_id];
            oi1 += p * V_tile[j][lane_id + 32];
        }

        __syncthreads();
    }

    // =========================
    // Final normalization
    // =========================

    O[i * D + lane_id] =
        oi0 / l;

    O[i * D + lane_id + 32] =
        oi1 / l;
}



#define CUDA_CHECK(call)                                      \
{                                                             \
    cudaError_t err = call;                                   \
    if (err != cudaSuccess) {                                 \
        cout << "CUDA error: " << cudaGetErrorString(err)     \
             << " at " << __FILE__ << ":" << __LINE__ << endl;\
        exit(1);                                              \
    }                                                         \
}



// host code
int main(){
  int N = 8192;
  int d = 64;

  size_t size = N * d * sizeof(float);

  //variables host + device
  float *h_Q, *h_K, *h_V, *h_O;
  float *d_Q, *d_K, *d_V, *d_O;

  // host memory allocation
  h_Q = (float*)malloc(size);
  h_K = (float*)malloc(size);
  h_V = (float*)malloc(size);
  h_O = (float*)malloc(size);

  // data initialization
  for(int i = 0; i < N* d; i++){
    h_Q[i] = static_cast<float>(rand()) / RAND_MAX;
    h_K[i] = static_cast<float>(rand()) / RAND_MAX;
    h_V[i] = static_cast<float>(rand()) / RAND_MAX;
  }

  // device memory allocation
  CUDA_CHECK(cudaMalloc((void**)&d_Q, size));
  CUDA_CHECK(cudaMalloc((void**)&d_K, size));
  CUDA_CHECK(cudaMalloc((void**)&d_V, size));
  CUDA_CHECK(cudaMalloc((void**)&d_O, size));

  //Copy data from host ->> to device
  CUDA_CHECK(cudaMemcpy(d_Q, h_Q , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_K, h_K , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_V, h_V , size , cudaMemcpyHostToDevice));


  // kernel config
  int threads = 256;
  int warps_per_block = threads / 32;
  int block = (N + warps_per_block - 1) / warps_per_block;

  // warmups
  for(int i = 0; i < 100; i++){
    flashAttention_v12<<<block , threads>>>(d_Q , d_K , d_V , d_O , N);
  }
  cudaDeviceSynchronize();

  //timing measurement
  cudaEvent_t start , stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);

  const int TIMED_RUNS = 100;
  for(int i = 0; i < TIMED_RUNS; i++){
    flashAttention_v12<<<block , threads>>>(d_Q, d_K, d_V, d_O , N);
  }

  cudaEventRecord(stop);
  cudaEventSynchronize(stop);


  // timing
  float ms = 0;
  cudaEventElapsedTime(&ms , start , stop);
  ms /= TIMED_RUNS;

  // FLOPs
  double flops = 4.0 * N * N * d;

  // Bytes (approx global memory traffic) or Throughput
  double bytes = 3.0 * N * N * sizeof(float);  // rough estimate

  // Metrics
  double gflops = (flops / (ms / 1000.0)) / 1e9;
  double bandwidth = (bytes / (ms / 1000.0)) / 1e9;
  double intensity = flops / bytes;

  cout << "Time: " << ms << " ms\n";
  cout << "GFLOPS: " << gflops << "\n";
  cout << "Memory Throughput (GB/s): " << bandwidth << "\n";
  cout << "Arithmetic Intensity: " << intensity << "\n";

  printf("METRIC,time_ms,%.4f\n", ms);
  printf("METRIC,gflops,%.2f\n", gflops);
  printf("METRIC,bandwidth,%.2f\n", bandwidth);
  printf("METRIC,intensity,%.2f\n", intensity);

  //cleanup
  cudaFree(d_Q);
  cudaFree(d_K);
  cudaFree(d_V);
  cudaFree(d_O);

  free(h_Q);
  free(h_K);
  free(h_V);
  free(h_O);

  return 0;
}

Writing flash_v12.cu


In [ ]:
!nvcc flash_v12.cu -o flash_v12 -arch=sm_75

In [ ]:
!./flash_v12

Time: 30.6747 ms
GFLOPS: 560.067
Memory Throughput (GB/s): 26.2531
Arithmetic Intensity: 21.3333
METRIC,time_ms,30.6747
METRIC,gflops,560.07
METRIC,bandwidth,26.25
METRIC,intensity,21.33


In [ ]:
result = subprocess.check_output([
    "ncu",
    "--launch-count", "1",
    "--launch-skip", "100",

    "--section", "WarpStateStats",
    "--section", "MemoryWorkloadAnalysis",
    "--section", "Occupancy",
    "--section", "SpeedOfLight",

    "./flash_v12"
], text=True, stderr=subprocess.STDOUT)

print(result)

==PROF== Connected to process 2738 (/content/flash_v12)
==PROF== Profiling "flashAttention_v12" - 0 (1/1): 0%....50%....100% - 12 passes
Time: 74.1239 ms
GFLOPS: 231.772
Memory Throughput (GB/s): 10.8643
Arithmetic Intensity: 21.3333
METRIC,time_ms,74.1239
METRIC,gflops,231.77
METRIC,bandwidth,10.86
METRIC,intensity,21.33
==PROF== Disconnected from process 2738
[2738] flash_v12@127.0.0.1
  flashAttention_v12(float *, float *, float *, float *, int) (1024, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- -------------
    Metric Name             Metric Unit  Metric Value
    ----------------------- ----------- -------------
    DRAM Frequency                  Ghz          5.00
    SM Frequency                    Mhz        585.00
    Elapsed Cycles                cycle    38,517,162
    Memory Throughput                 %         89.31
    DRAM Throughput                   %          0.28
    Dura

In [ ]:
import subprocess
import wandb

# persistent iteration counter across reruns
if "iteration" not in globals():
    iteration = 0

iteration += 1

# only init if not already active
if wandb.run is None:
    wandb.init(
        project="flashattention-cuda",
        name="v10-profile",
        resume="allow",
        id="v10-profile"
    )

# compile CUDA kernel
subprocess.run(
    ["nvcc", "flash_v10.cu", "-o", "flash_v10", "-arch=sm_75"],
    check=True
)

# execute kernel
result = subprocess.check_output(["./flash_v10"], text=True)
print(result)

metrics = {}

for line in result.splitlines():
    if line.startswith("METRIC"):
        _, key, value = line.split(",")
        try:
            metrics[key] = float(value)
        except:
            metrics[key] = value

metrics["iteration"] = iteration
print(metrics)

# log with explicit step so x-axis is always clean
wandb.log(metrics, step=iteration)



Time: 36.3342 ms
GFLOPS: 472.829
Memory Throughput (GB/s): 22.1639
Arithmetic Intensity: 21.3333
METRIC,time_ms,36.3342
METRIC,gflops,472.83
METRIC,bandwidth,22.16
METRIC,intensity,21.33

{'time_ms': 36.3342, 'gflops': 472.83, 'bandwidth': 22.16, 'intensity': 21.33, 'iteration': 1}


### PHASE 13 - FLASHATTENTION - v13


In [ ]:
%%writefile flash_13.cu
#include<iostream>
#include<cuda_runtime.h>
using namespace std;

#define TILE_SIZE 32
#define D 64

__global__ void flashAttention_v13(float *Q, float *K, float *V, float *O , int N){
  //warp identification
  int warp_id = threadIdx.x >> 5;
  int lane_id = threadIdx.x & 31;

  int warps_per_block = blockDim.x >> 5;

  int i = blockIdx.x * warps_per_block + warp_id;

  if(i >= N) return;

  // shared memory
  __shared__ float K_tile[TILE_SIZE][D];
  __shared__ float V_tile[TILE_SIZE][D];

  float scale = rsqrtf((float)D);

  // Register loading
  float qi0 = Q[i * D + lane_id] * scale;
  float qi1 = Q[i * D + lane_id + 32] * scale;

  // outputs accumulators
  float oi0 = 0.0f;
  float oi1 = 0.0f;

  // online softmax variables
  float m = -INFINITY;
  float l = 0.f;

  // main loop  =======================
  for(int tile_start = 0; tile_start < N; tile_start += TILE_SIZE){

    // vectorized tile loading
    int tid = threadIdx.x;

    for(int idx = tid; idx < (TILE_SIZE * D) / 4; idx += blockDim.x){
      int base = idx << 2;
      int row = base >> 6;
      int col = base & 63;

      if(tile_start + row < N){
        float4 kv = reinterpret_cast<float4*>(&K[(tile_start + row) * D + col])[0];
        float4 vv = reinterpret_cast<float4*>(&V[(tile_start + row) * D + col])[0];

        reinterpret_cast<float4*>(&K_tile[row][col])[0] = kv;
        reinterpret_cast<float4*>(&V_tile[row][col])[0] = vv;
      }
    }

    __syncthreads();

    int valid_j = min(TILE_SIZE , N - tile_start);
    int full_j = (valid_j >> 2) << 2;

    // full tile path
    for(int j = 0; j < full_j; j += 4){
      // register cache K values
      float k00 = K_tile[j][lane_id];
      float k01 = K_tile[j][lane_id + 32];

      float k10 = K_tile[j+1][lane_id];
      float k11 = K_tile[j+1][lane_id + 32];

      float k20 = K_tile[j+2][lane_id];
      float k21 = K_tile[j+2][lane_id + 32];

      float k30 = K_tile[j+3][lane_id];
      float k31 = K_tile[j+3][lane_id + 32];

      //Register Cache V values ---------------
      float v00 = V_tile[j][lane_id];
      float v01 = V_tile[j][lane_id + 32];

      float v10 = V_tile[j+1][lane_id];
      float v11 = V_tile[j+1][lane_id + 32];

      float v20 = V_tile[j+2][lane_id];
      float v21 = V_tile[j+2][lane_id + 32];

      float v30 = V_tile[j+3][lane_id];
      float v31 = V_tile[j+3][lane_id + 32];


      // dot products pure register computes

      float s0 = qi0 * k00 + qi1 * k01;
      float s1 = qi0 * k10 + qi1 * k11;
      float s2 = qi0 * k20 + qi1 * k21;
      float s3 = qi0 * k30 + qi1 * k31;

      // warp reduction
      #pragma unroll
      for(int offset = 16; offset > 0; offset >>= 1){
        s0 += __shfl_down_sync(0xffffffff, s0 , offset);
        s1 += __shfl_down_sync(0xffffffff, s1 , offset);
        s2 += __shfl_down_sync(0xffffffff, s2 , offset);
        s3 += __shfl_down_sync(0xffffffff, s3 , offset);
      }

      s0 = __shfl_sync(0xffffffff, s0 , 0);
      s1 = __shfl_sync(0xffffffff, s1 , 0);
      s2 = __shfl_sync(0xffffffff, s2 , 0);
      s3 = __shfl_sync(0xffffffff, s3 , 0);

      // online softmax

      float m_old = m;
      float local_max = fmaxf(fmaxf(s0 , s1),
                              fmaxf(s2 , s3));
      float m_new = fmaxf(m , local_max);

      // computing expon. ONCE
      float p0 = __expf(s0 - m_new);
      float p1 = __expf(s1 - m_new);
      float p2 = __expf(s2 - m_new);
      float p3 = __expf(s3 - m_new);

      float scale_old = __expf(m_old - m_new);
      l = l * scale_old + p0 + p1 + p2 + p3;
      m = m_new;

      // rescale outputs
      oi0 *= scale_old;
      oi1 *= scale_old;


      // outputs accumalation pure register
      oi0 += p0 * v00 + p1 * v10 + p2 * v20 + p3 * v30;
      oi1 += p0 * v01 + p1 * v11 + p2 * v21 + p3 * v31;
    }

    // TAIL path
    for(int j = full_j; j < valid_j; j++){

      // register cache
      float k0 = K_tile[j][lane_id];
      float k1 = K_tile[j][lane_id + 32];

      float v0 = V_tile[j][lane_id];
      float v1 = V_tile[j][lane_id + 32];

      // DOT products
      float s = qi0 * k0 + qi1 * k1;

      #pragma unroll
      for(int offset = 16; offset > 0 ; offset >>= 1){
        s += __shfl_down_sync(0xffffffff, s , offset);
      }
      s = __shfl_sync(0xffffffff, s , 0);

      // online softmax
      float m_old = m;
      float m_new = fmaxf(m , s);
      float p = __expf(s - m_new);
      float scale_old = __expf(m_old - m_new);
      l = l * scale_old + p;
      m = m_new;

      oi0 *= scale_old;
      oi1 *= scale_old;

      // Accumulate
      oi0 += p * v0;
      oi1 += p * v1;
    }
    __syncthreads();

  }

  O[i * D + lane_id] = oi0 / l;

  O[i * D + lane_id + 32] = oi1 / l;
}





#define CUDA_CHECK(call)                                      \
{                                                             \
    cudaError_t err = call;                                   \
    if (err != cudaSuccess) {                                 \
        cout << "CUDA error: " << cudaGetErrorString(err)     \
             << " at " << __FILE__ << ":" << __LINE__ << endl;\
        exit(1);                                              \
    }                                                         \
}



// host code
int main(){
  int N = 8192;
  int d = 64;

  size_t size = N * d * sizeof(float);

  //variables host + device
  float *h_Q, *h_K, *h_V, *h_O;
  float *d_Q, *d_K, *d_V, *d_O;

  // host memory allocation
  h_Q = (float*)malloc(size);
  h_K = (float*)malloc(size);
  h_V = (float*)malloc(size);
  h_O = (float*)malloc(size);

  // data initialization
  for(int i = 0; i < N* d; i++){
    h_Q[i] = static_cast<float>(rand()) / RAND_MAX;
    h_K[i] = static_cast<float>(rand()) / RAND_MAX;
    h_V[i] = static_cast<float>(rand()) / RAND_MAX;
  }

  // device memory allocation
  CUDA_CHECK(cudaMalloc((void**)&d_Q, size));
  CUDA_CHECK(cudaMalloc((void**)&d_K, size));
  CUDA_CHECK(cudaMalloc((void**)&d_V, size));
  CUDA_CHECK(cudaMalloc((void**)&d_O, size));

  //Copy data from host ->> to device
  CUDA_CHECK(cudaMemcpy(d_Q, h_Q , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_K, h_K , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_V, h_V , size , cudaMemcpyHostToDevice));


  // kernel config
  int threads = 256;
  int warps_per_block = threads / 32;
  int block = (N + warps_per_block - 1) / warps_per_block;

  // warmups
  for(int i = 0; i < 100; i++){
    flashAttention_v13<<<block , threads>>>(d_Q , d_K , d_V , d_O , N);
  }
  cudaDeviceSynchronize();

  //timing measurement
  cudaEvent_t start , stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);

  const int TIMED_RUNS = 100;
  for(int i = 0; i < TIMED_RUNS; i++){
    flashAttention_v13<<<block , threads>>>(d_Q, d_K, d_V, d_O , N);
  }

  cudaEventRecord(stop);
  cudaEventSynchronize(stop);


  // timing
  float ms = 0;
  cudaEventElapsedTime(&ms , start , stop);
  ms /= TIMED_RUNS;

  // FLOPs
  double flops = 4.0 * N * N * d;

  // Bytes (approx global memory traffic) or Throughput
  double bytes = 3.0 * N * N * sizeof(float);  // rough estimate

  // Metrics
  double gflops = (flops / (ms / 1000.0)) / 1e9;
  double bandwidth = (bytes / (ms / 1000.0)) / 1e9;
  double intensity = flops / bytes;

  cout << "Time: " << ms << " ms\n";
  cout << "GFLOPS: " << gflops << "\n";
  cout << "Memory Throughput (GB/s): " << bandwidth << "\n";
  cout << "Arithmetic Intensity: " << intensity << "\n";

  printf("METRIC,time_ms,%.4f\n", ms);
  printf("METRIC,gflops,%.2f\n", gflops);
  printf("METRIC,bandwidth,%.2f\n", bandwidth);
  printf("METRIC,intensity,%.2f\n", intensity);

  //cleanup
  cudaFree(d_Q);
  cudaFree(d_K);
  cudaFree(d_V);
  cudaFree(d_O);

  free(h_Q);
  free(h_K);
  free(h_V);
  free(h_O);

  return 0;
}

Overwriting flash_13.cu


In [ ]:
!nvcc flash_13.cu -o flash_13 -arch=sm_75

In [ ]:
!./flash_13

Time: 29.9365 ms
GFLOPS: 573.878
Memory Throughput (GB/s): 26.9005
Arithmetic Intensity: 21.3333
METRIC,time_ms,29.9365
METRIC,gflops,573.88
METRIC,bandwidth,26.90
METRIC,intensity,21.33


In [ ]:
import subprocess

result = subprocess.check_output([
    "ncu",
    "--launch-count", "1",
    "--launch-skip", "100",

    "--section", "WarpStateStats",
    "--section", "MemoryWorkloadAnalysis",
    "--section", "Occupancy",
    "--section", "SpeedOfLight",

    "./flash_13"
], text=True, stderr=subprocess.STDOUT)

print(result)

==PROF== Connected to process 3186 (/content/flash_13)
==PROF== Profiling "flashAttention_v13" - 0 (1/1): 0%....50%....100% - 12 passes
Time: 73.1385 ms
GFLOPS: 234.895
Memory Throughput (GB/s): 11.0107
Arithmetic Intensity: 21.3333
METRIC,time_ms,73.1385
METRIC,gflops,234.90
METRIC,bandwidth,11.01
METRIC,intensity,21.33
==PROF== Disconnected from process 3186
[3186] flash_13@127.0.0.1
  flashAttention_v13(float *, float *, float *, float *, int) (1024, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- -------------
    Metric Name             Metric Unit  Metric Value
    ----------------------- ----------- -------------
    DRAM Frequency                  Ghz          5.00
    SM Frequency                    Mhz        585.00
    Elapsed Cycles                cycle    38,506,001
    Memory Throughput                 %         89.31
    DRAM Throughput                   %          0.28
    Durati

### PAHSE-14 FLASHATTENTION-V14
* Here we are gonna solve the MIO problems:
  * Shared memory instruction pressure by register preloading
  * expf tp exp2f
  * dealing with Shuffles
  * balancing the ILP

In [ ]:
%%writefile flash_14.cu
#include<iostream>
#include<cuda_runtime.h>
using namespace std;

#define TILE_SIZE 32
#define D 64

__global__ void flashAttention_v14(float *Q , float *K , float *V, float *O, int N){

  //warp identification
  int warp_id = threadIdx.x >> 5;
  int lane_id = threadIdx.x & 31;
  int warps_per_block = blockDim.x >> 5;

  int i = blockIdx.x * warps_per_block + warp_id;

  if( i >= N) return;

  // shared memory tile
  __shared__ float K_tile[TILE_SIZE][D];
  __shared__ float V_tile[TILE_SIZE][D];

  // constants inside the registers
  const float scale = rsqrtf((float)D);
  const float LOG2E = 1.4426950408f;  // converts the expf - > exp2f

  // load Q into registers
  const float qi0 = Q[i * D + lane_id] * scale;
  const float qi1 = Q[i * D + lane_id + 32] * scale;

  // output accumulators
  float oi0 = 0.f;
  float oi1 = 0.f;

  // online softmax
  float m = -INFINITY;
  float l = 0.0f;


  //main Loop
  for(int tile_start = 0; tile_start < N; tile_start += TILE_SIZE){
    const int tid = threadIdx.x;


    for(int idx = tid; idx < (TILE_SIZE * D) / 4; idx += blockDim.x){
      const int base = idx << 2;
      const int row  = base >> 6;
      const int col  = base & 63;
      if(tile_start + row < N){
        reinterpret_cast<float4*>(&K_tile[row][col])[0] = reinterpret_cast<const float4*>(&K[(tile_start + row) * D + col])[0];
        reinterpret_cast<float4*>(&V_tile[row][col])[0] = reinterpret_cast<const float4*>(&V[(tile_start + row) * D + col])[0];
      }
    }
    __syncthreads();

    const int valid_j = min(TILE_SIZE , N - tile_start);
    const int full_j = (valid_j >> 2) << 2;  // largest multiple of 4

    for (int j = 0; j < full_j; j += 4){
      const float2 k0 = reinterpret_cast<float2*>(&K_tile[j  ][lane_id * 2])[0];
      const float2 k1 = reinterpret_cast<float2*>(&K_tile[j+1][lane_id * 2])[0];
      const float2 k2 = reinterpret_cast<float2*>(&K_tile[j+2][lane_id * 2])[0];
      const float2 k3 = reinterpret_cast<float2*>(&K_tile[j+3][lane_id * 2])[0];


      const float2 v0 = reinterpret_cast<float2*>(&V_tile[j  ][lane_id * 2])[0];
      const float2 v1 = reinterpret_cast<float2*>(&V_tile[j+1][lane_id * 2])[0];
      const float2 v2 = reinterpret_cast<float2*>(&V_tile[j+2][lane_id * 2])[0];
      const float2 v3 = reinterpret_cast<float2*>(&V_tile[j+3][lane_id * 2])[0];


      // 4 independent FMAs → full ILP
      float s0 = qi0 * k0.x + qi1 * k0.y;
      float s1 = qi0 * k1.x + qi1 * k1.y;
      float s2 = qi0 * k2.x + qi1 * k2.y;
      float s3 = qi0 * k3.x + qi1 * k3.y;


      #pragma unroll
      for (int offset = 16; offset > 0; offset >>= 1) {
        s0 += __shfl_down_sync(0xffffffff, s0, offset);
        s1 += __shfl_down_sync(0xffffffff, s1, offset);
        s2 += __shfl_down_sync(0xffffffff, s2, offset);
        s3 += __shfl_down_sync(0xffffffff, s3, offset);
      }
      s0 = __shfl_sync(0xffffffff, s0, 0);
      s1 = __shfl_sync(0xffffffff, s1, 0);
      s2 = __shfl_sync(0xffffffff, s2, 0);
      s3 = __shfl_sync(0xffffffff, s3, 0);


      const float m_old     = m;
      const float local_max = fmaxf(fmaxf(s0, s1), fmaxf(s2, s3));
      const float m_new     = fmaxf(m, local_max);


      const float p0      = exp2f((s0    - m_new) * LOG2E);   // independent
      const float p1      = exp2f((s1    - m_new) * LOG2E);   // independent
      const float p2      = exp2f((s2    - m_new) * LOG2E);   // independent
      const float p3      = exp2f((s3    - m_new) * LOG2E);   // independent
      const float p_scale = exp2f((m_old - m_new) * LOG2E);   // rescales old state

      l    = l   * p_scale + p0 + p1 + p2 + p3;
      m    = m_new;

      oi0  = oi0 * p_scale + p0 * v0.x + p1 * v1.x + p2 * v2.x + p3 * v3.x;
      oi1  = oi1 * p_scale + p0 * v0.y + p1 * v1.y + p2 * v2.y + p3 * v3.y;
    }

    for (int j = full_j; j < valid_j; j++){
      const float2 k = reinterpret_cast<float2*>(&K_tile[j][lane_id * 2])[0];
      const float2 v = reinterpret_cast<float2*>(&V_tile[j][lane_id * 2])[0];

      float s = qi0 * k.x + qi1 * k.y;

      #pragma unroll
      for (int offset = 16; offset > 0; offset >>= 1)
        s += __shfl_down_sync(0xffffffff, s, offset);
      s = __shfl_sync(0xffffffff, s, 0);

      const float m_old   = m;
      const float m_new   = fmaxf(m, s);
      const float p       = exp2f((s     - m_new) * LOG2E);
      const float p_scale = exp2f((m_old - m_new) * LOG2E);

      l    = l   * p_scale + p;
      m    = m_new;

      oi0  = oi0 * p_scale + p * v.x;
      oi1  = oi1 * p_scale + p * v.y;

    }
    __syncthreads();

  }

  const float inv_l = 1.f / l;
  O[i * D + lane_id * 2    ] = oi0 * inv_l;
  O[i * D + lane_id * 2 + 1] = oi1 * inv_l;
}





#define CUDA_CHECK(call)                                      \
{                                                             \
    cudaError_t err = call;                                   \
    if (err != cudaSuccess) {                                 \
        cout << "CUDA error: " << cudaGetErrorString(err)     \
             << " at " << __FILE__ << ":" << __LINE__ << endl;\
        exit(1);                                              \
    }                                                         \
}



// host code
int main(){
  int N = 8192;
  int d = 64;

  size_t size = N * d * sizeof(float);

  //variables host + device
  float *h_Q, *h_K, *h_V, *h_O;
  float *d_Q, *d_K, *d_V, *d_O;

  // host memory allocation
  h_Q = (float*)malloc(size);
  h_K = (float*)malloc(size);
  h_V = (float*)malloc(size);
  h_O = (float*)malloc(size);

  // data initialization
  for(int i = 0; i < N* d; i++){
    h_Q[i] = static_cast<float>(rand()) / RAND_MAX;
    h_K[i] = static_cast<float>(rand()) / RAND_MAX;
    h_V[i] = static_cast<float>(rand()) / RAND_MAX;
  }

  // device memory allocation
  CUDA_CHECK(cudaMalloc((void**)&d_Q, size));
  CUDA_CHECK(cudaMalloc((void**)&d_K, size));
  CUDA_CHECK(cudaMalloc((void**)&d_V, size));
  CUDA_CHECK(cudaMalloc((void**)&d_O, size));

  //Copy data from host ->> to device
  CUDA_CHECK(cudaMemcpy(d_Q, h_Q , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_K, h_K , size , cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_V, h_V , size , cudaMemcpyHostToDevice));


  // kernel config
  int threads = 256;
  int warps_per_block = threads / 32;
  int block = (N + warps_per_block - 1) / warps_per_block;

  // warmups
  for(int i = 0; i < 100; i++){
    flashAttention_v14<<<block , threads>>>(d_Q , d_K , d_V , d_O , N);
  }
  cudaDeviceSynchronize();

  //timing measurement
  cudaEvent_t start , stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);

  const int TIMED_RUNS = 100;
  for(int i = 0; i < TIMED_RUNS; i++){
    flashAttention_v14<<<block , threads>>>(d_Q, d_K, d_V, d_O , N);
  }

  cudaEventRecord(stop);
  cudaEventSynchronize(stop);


  // timing
  float ms = 0;
  cudaEventElapsedTime(&ms , start , stop);
  ms /= TIMED_RUNS;

  // FLOPs
  double flops = 4.0 * N * N * d;

  // Bytes (approx global memory traffic) or Throughput
  double bytes = 3.0 * N * N * sizeof(float);  // rough estimate

  // Metrics
  double gflops = (flops / (ms / 1000.0)) / 1e9;
  double bandwidth = (bytes / (ms / 1000.0)) / 1e9;
  double intensity = flops / bytes;

  cout << "Time: " << ms << " ms\n";
  cout << "GFLOPS: " << gflops << "\n";
  cout << "Memory Throughput (GB/s): " << bandwidth << "\n";
  cout << "Arithmetic Intensity: " << intensity << "\n";

  printf("METRIC,time_ms,%.4f\n", ms);
  printf("METRIC,gflops,%.2f\n", gflops);
  printf("METRIC,bandwidth,%.2f\n", bandwidth);
  printf("METRIC,intensity,%.2f\n", intensity);

  //cleanup
  cudaFree(d_Q);
  cudaFree(d_K);
  cudaFree(d_V);
  cudaFree(d_O);

  free(h_Q);
  free(h_K);
  free(h_V);
  free(h_O);

  return 0;
}

Writing flash_14.cu


In [ ]:
!nvcc flash_14.cu -o flash_14 -arch=sm_75

In [ ]:
!./flash_14

Time: 29.39 ms
GFLOPS: 584.547
Memory Throughput (GB/s): 27.4007
Arithmetic Intensity: 21.3333
METRIC,time_ms,29.3900
METRIC,gflops,584.55
METRIC,bandwidth,27.40
METRIC,intensity,21.33


In [ ]:
import subprocess

result = subprocess.check_output([
    "ncu",
    "--launch-count", "1",
    "--launch-skip", "100",

    "--section", "WarpStateStats",
    "--section", "MemoryWorkloadAnalysis",
    "--section", "Occupancy",
    "--section", "SpeedOfLight",

    "./flash_14"
], text=True, stderr=subprocess.STDOUT)

print(result)

==PROF== Connected to process 4680 (/content/flash_14)
==PROF== Profiling "flashAttention_v14" - 0 (1/1): 0%....50%....100% - 12 passes
Time: 73.32 ms
GFLOPS: 234.314
Memory Throughput (GB/s): 10.9834
Arithmetic Intensity: 21.3333
METRIC,time_ms,73.3200
METRIC,gflops,234.31
METRIC,bandwidth,10.98
METRIC,intensity,21.33
==PROF== Disconnected from process 4680
[4680] flash_14@127.0.0.1
  flashAttention_v14(float *, float *, float *, float *, int) (1024, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- -------------
    Metric Name             Metric Unit  Metric Value
    ----------------------- ----------- -------------
    DRAM Frequency                  Ghz          5.00
    SM Frequency                    Mhz        585.00
    Elapsed Cycles                cycle    38,110,950
    Memory Throughput                 %         72.64
    DRAM Throughput                   %          0.29
    Duration

In [ ]:
%%writefile flash_15.cu
#include<iostream>
#include<cuda_runtime.h>
using namespace std;

#define TILE_SIZE 32
#define D 64



__global__ void __launch_bounds__(256, 3)
flashAttention_v15(float* __restrict__ Q,
                   float* __restrict__ K,
                   float* __restrict__ V,
                   float* __restrict__ O,
                   int N)
{
    const int warp_id         = threadIdx.x >> 5;
    const int lane_id         = threadIdx.x & 31;
    const int warps_per_block = blockDim.x >> 5;

    const int i = blockIdx.x * warps_per_block + warp_id;
    if (i >= N) return;

    // 32x64x4 = 8KB each
    // total = 16KB shared memory
    __shared__ float K_tile[TILE_SIZE][D];
    __shared__ float V_tile[TILE_SIZE][D];

    const float scale = rsqrtf((float)D);
    const float LOG2E = 1.4426950408f;

    // each lane owns 2 dimensions
    const float qi0 = Q[i * D + lane_id * 2    ] * scale;
    const float qi1 = Q[i * D + lane_id * 2 + 1] * scale;

    float oi0 = 0.f;
    float oi1 = 0.f;

    float m = -INFINITY;
    float l = 0.f;

    for (int tile_start = 0; tile_start < N; tile_start += TILE_SIZE)
    {
        // vectorized cooperative load
        for (int idx = threadIdx.x;
             idx < (TILE_SIZE * D) / 4;
             idx += blockDim.x)
        {
            const int vec_idx = idx;

            const int row = vec_idx >> 4;          // /16
            const int col = (vec_idx & 15) << 2;  // *4

            if (tile_start + row < N)
            {
                reinterpret_cast<float4*>(&K_tile[row][col])[0] =
                    reinterpret_cast<const float4*>(
                        &K[(tile_start + row) * D + col])[0];

                reinterpret_cast<float4*>(&V_tile[row][col])[0] =
                    reinterpret_cast<const float4*>(
                        &V[(tile_start + row) * D + col])[0];
            }
        }

        __syncthreads();

        const int valid_j = min(TILE_SIZE, N - tile_start);
        const int full_j  = (valid_j >> 2) << 2;

        // process 4 rows at a time
        for (int j = 0; j < full_j; j += 4)
        {
            // rolling register fragments
            float2 k0 =
                reinterpret_cast<float2*>(&K_tile[j][lane_id * 2])[0];
            float2 k1 =
                reinterpret_cast<float2*>(&K_tile[j + 1][lane_id * 2])[0];
            float2 k2 =
                reinterpret_cast<float2*>(&K_tile[j + 2][lane_id * 2])[0];
            float2 k3 =
                reinterpret_cast<float2*>(&K_tile[j + 3][lane_id * 2])[0];

            float2 v0 =
                reinterpret_cast<float2*>(&V_tile[j][lane_id * 2])[0];
            float2 v1 =
                reinterpret_cast<float2*>(&V_tile[j + 1][lane_id * 2])[0];
            float2 v2 =
                reinterpret_cast<float2*>(&V_tile[j + 2][lane_id * 2])[0];
            float2 v3 =
                reinterpret_cast<float2*>(&V_tile[j + 3][lane_id * 2])[0];

            // partial dot products
            float s0 = qi0 * k0.x + qi1 * k0.y;
            float s1 = qi0 * k1.x + qi1 * k1.y;
            float s2 = qi0 * k2.x + qi1 * k2.y;
            float s3 = qi0 * k3.x + qi1 * k3.y;

            // packed shuffle reduction
            union Pack64 {
                uint64_t u64;
                struct { float a, b; };
            };

            Pack64 p01, p23;

            p01.a = s0;
            p01.b = s1;

            p23.a = s2;
            p23.b = s3;

            #pragma unroll
            for (int offset = 16; offset > 0; offset >>= 1)
            {
                Pack64 tmp01, tmp23;

                tmp01.u64 =
                    __shfl_down_sync(0xffffffff, p01.u64, offset);

                tmp23.u64 =
                    __shfl_down_sync(0xffffffff, p23.u64, offset);

                p01.a += tmp01.a;
                p01.b += tmp01.b;

                p23.a += tmp23.a;
                p23.b += tmp23.b;
            }

            p01.u64 = __shfl_sync(0xffffffff, p01.u64, 0);
            p23.u64 = __shfl_sync(0xffffffff, p23.u64, 0);

            const float rs0 = p01.a;
            const float rs1 = p01.b;
            const float rs2 = p23.a;
            const float rs3 = p23.b;

            // online softmax
            const float m_old     = m;
            const float local_max =
                fmaxf(fmaxf(rs0, rs1), fmaxf(rs2, rs3));

            const float m_new   = fmaxf(m_old, local_max);
            const float p_scale =
                exp2f((m_old - m_new) * LOG2E);

            const float ep0 =
                exp2f((rs0 - m_new) * LOG2E);

            const float ep1 =
                exp2f((rs1 - m_new) * LOG2E);

            const float ep2 =
                exp2f((rs2 - m_new) * LOG2E);

            const float ep3 =
                exp2f((rs3 - m_new) * LOG2E);

            oi0 = oi0 * p_scale
                + ep0 * v0.x
                + ep1 * v1.x
                + ep2 * v2.x
                + ep3 * v3.x;

            oi1 = oi1 * p_scale
                + ep0 * v0.y
                + ep1 * v1.y
                + ep2 * v2.y
                + ep3 * v3.y;

            l = l * p_scale + ep0 + ep1 + ep2 + ep3;

            m = m_new;
        }

        // tail handling
        for (int j = full_j; j < valid_j; j++)
        {
            float2 k =
                reinterpret_cast<float2*>(&K_tile[j][lane_id * 2])[0];

            float2 v =
                reinterpret_cast<float2*>(&V_tile[j][lane_id * 2])[0];

            float s = qi0 * k.x + qi1 * k.y;

            #pragma unroll
            for (int offset = 16; offset > 0; offset >>= 1)
                s += __shfl_down_sync(0xffffffff, s, offset);

            s = __shfl_sync(0xffffffff, s, 0);

            const float m_old   = m;
            const float m_new   = fmaxf(m_old, s);

            const float p_scale =
                exp2f((m_old - m_new) * LOG2E);

            const float ep =
                exp2f((s - m_new) * LOG2E);

            oi0 = oi0 * p_scale + ep * v.x;
            oi1 = oi1 * p_scale + ep * v.y;

            l = l * p_scale + ep;

            m = m_new;
        }

        __syncthreads();
    }

    const float inv_l = __fdividef(1.f, l);

    O[i * D + lane_id * 2    ] = oi0 * inv_l;
    O[i * D + lane_id * 2 + 1] = oi1 * inv_l;
}



#define CUDA_CHECK(call)                                      \
{                                                             \
    cudaError_t err = call;                                   \
    if (err != cudaSuccess) {                                 \
        cout << "CUDA error: " << cudaGetErrorString(err)     \
             << " at " << __FILE__ << ":" << __LINE__ << endl;\
        exit(1);                                              \
    }                                                         \
}


int main(){
    int N = 8192;
    int d = 64;

    size_t size = N * d * sizeof(float);

    float *h_Q, *h_K, *h_V, *h_O;
    float *d_Q, *d_K, *d_V, *d_O;

    h_Q = (float*)malloc(size);
    h_K = (float*)malloc(size);
    h_V = (float*)malloc(size);
    h_O = (float*)malloc(size);

    for(int i = 0; i < N * d; i++){
        h_Q[i] = static_cast<float>(rand()) / RAND_MAX;
        h_K[i] = static_cast<float>(rand()) / RAND_MAX;
        h_V[i] = static_cast<float>(rand()) / RAND_MAX;
    }

    CUDA_CHECK(cudaMalloc((void**)&d_Q, size));
    CUDA_CHECK(cudaMalloc((void**)&d_K, size));
    CUDA_CHECK(cudaMalloc((void**)&d_V, size));
    CUDA_CHECK(cudaMalloc((void**)&d_O, size));

    CUDA_CHECK(cudaMemcpy(d_Q, h_Q, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_K, h_K, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_V, h_V, size, cudaMemcpyHostToDevice));

    // ── kernel config ──────────────────────────────────────────
    // Each warp handles 2 rows → need N/2 warps total
    int threads          = 256;
    int warps_per_block  = threads / 32;
    int total_warps      = (N / 2 + 1);           // warps needed
    int blocks           = (total_warps + warps_per_block - 1) / warps_per_block;

    // warmup
    for(int i = 0; i < 100; i++)
        flashAttention_v15<<<blocks, threads>>>(d_Q, d_K, d_V, d_O, N);
    cudaDeviceSynchronize();

    // timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    const int TIMED_RUNS = 100;
    for(int i = 0; i < TIMED_RUNS; i++)
        flashAttention_v15<<<blocks, threads>>>(d_Q, d_K, d_V, d_O, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);
    ms /= TIMED_RUNS;

    double flops     = 4.0 * N * N * d;
    double bytes     = 3.0 * N * N * sizeof(float);
    double gflops    = (flops / (ms / 1000.0)) / 1e9;
    double bandwidth = (bytes / (ms / 1000.0)) / 1e9;
    double intensity = flops / bytes;

    cout << "Time: "                      << ms        << " ms\n";
    cout << "GFLOPS: "                    << gflops    << "\n";
    cout << "Memory Throughput (GB/s): "  << bandwidth << "\n";
    cout << "Arithmetic Intensity: "      << intensity << "\n";

    printf("METRIC,time_ms,%.4f\n",  ms);
    printf("METRIC,gflops,%.2f\n",   gflops);
    printf("METRIC,bandwidth,%.2f\n",bandwidth);
    printf("METRIC,intensity,%.2f\n", intensity);

    cudaFree(d_Q); cudaFree(d_K); cudaFree(d_V); cudaFree(d_O);
    free(h_Q);     free(h_K);     free(h_V);     free(h_O);

    return 0;
}

Overwriting flash_15.cu


In [ ]:
!nvcc flash_15.cu -o flash_15 -arch=sm_75

In [ ]:
!./flash_15

Time: 15.3984 ms
GFLOPS: 1115.69
Memory Throughput (GB/s): 52.2981
Arithmetic Intensity: 21.3333
METRIC,time_ms,15.3984
METRIC,gflops,1115.69
METRIC,bandwidth,52.30
METRIC,intensity,21.33


In [ ]:
import subprocess

result = subprocess.check_output([
    "ncu",
    "--launch-count", "1",
    "--launch-skip", "100",

    "--section", "WarpStateStats",
    "--section", "MemoryWorkloadAnalysis",
    "--section", "Occupancy",
    "--section", "SpeedOfLight",

    "./flash_15"
], text=True, stderr=subprocess.STDOUT)

print(result)

==PROF== Connected to process 3030 (/content/flash_15)
==PROF== Profiling "flashAttention_v15" - 0 (1/1): 0%....50%....100% - 12 passes
Time: 54.7182 ms
GFLOPS: 313.97
Memory Throughput (GB/s): 14.7173
Arithmetic Intensity: 21.3333
METRIC,time_ms,54.7182
METRIC,gflops,313.97
METRIC,bandwidth,14.72
METRIC,intensity,21.33
==PROF== Disconnected from process 3030
[3030] flash_15@127.0.0.1
  flashAttention_v15(float *, float *, float *, float *, int) (513, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- -------------
    Metric Name             Metric Unit  Metric Value
    ----------------------- ----------- -------------
    DRAM Frequency                  Ghz          5.00
    SM Frequency                    Mhz        585.00
    Elapsed Cycles                cycle    19,148,377
    Memory Throughput                 %         72.44
    DRAM Throughput                   %          0.28
    Duration